In [ ]:
import random
import math
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image


# 距離の計算

def tour_length(tour, cities):
    total = 0.0
    n = len(tour)
    for i in range(n):
        a = cities[tour[i]]
        b = cities[tour[(i + 1) % n]]
        total += math.dist(a, b)
    return total



# トーナメント選択
def tournament_select(population, fitnesses, k=3):
    indices = random.sample(range(len(population)), k)
    best_idx = min(indices, key=lambda i: fitnesses[i])  
    return population[best_idx]


# OX 交叉
def crossover_ox(parent1, parent2):
    n = len(parent1)
    c1, c2 = sorted(random.sample(range(n), 2))
    
    child = [None] * n
    child[c1:c2+1] = parent1[c1:c2+1]
    
    pos = (c2 + 1) % n
    for gene in parent2:
        if gene not in child:
            child[pos] = gene
            pos = (pos + 1) % n

    assert None not in child
    return child


# 突然変異
def mutate_swap(tour, mutation_rate):
    if random.random() < mutation_rate:
        i, j = random.sample(range(len(tour)), 2)
        tour[i], tour[j] = tour[j], tour[i]


# GA本体
def ga_tsp(
    num_cities,
    pop_size,
    generations,
    crossover_rate,
    mutation_rate,
    seed,
):
    # 都市座標（ジブチ）
    cities = np.array([
        [11003.611100, 42102.500000],
        [11108.611100, 42373.888900],
        [11133.333300, 42885.833300],
        [11155.833300, 42712.500000],
        [11183.333300, 42933.333300],
        [11297.500000, 42853.333300],
        [11310.277800, 42929.444400],
        [11416.666700, 42983.333300],
        [11423.888900, 43000.277800],
        [11438.333300,42057.222200],
        [11461.111100, 43252.777800],
        [11485.555600, 43187.222200],
        [11503.055600, 42855.277800],
        [11511.388900, 42106.388900],
        [11522.222200, 42841.944400],
        [11569.444400, 43136.666700],
        [11583.333300, 43150.000000],
        [11595.000000, 43148.055600],
        [11600.000000, 43150.000000],
        [11690.555600, 42686.666700],
        [11715.833300, 41836.111100],
        [11751.111100, 42814.444400],
        [11770.277800, 42651.944400],
        [11785.277800, 42884.444400],
        [11822.777800, 42673.611100],
        [11846.944400, 42660.555600],
        [11963.055600, 43290.555600],
        [11973.055600, 43026.111100],
        [12058.333300, 42195.555600],
        [12149.444400, 42477.500000],
        [12286.944400, 43355.555600],
        [12300.000000, 42433.333300],
        [12355.833300, 43156.388900],
        [12363.333300, 43189.166700],
        [12372.777800, 42711.388900],
        [12386.666700, 43334.722200],
        [12421.666700, 42895.555600],
        [12645.000000, 42973.333300],
    ])

    # 保存フォルダ準備
    os.makedirs("frames", exist_ok=True)

    random.seed(seed)
    np.random.seed(seed)

    base = list(range(num_cities))
    population = [random.sample(base, num_cities) for _ in range(pop_size)]

    best_tour = None
    best_len = float("inf")
    best_history = []  
    frame_paths = []   # GIF用画像パス

    for gen in range(generations):
        fitnesses = [tour_length(ind, cities) for ind in population]

        # ベスト更新
        for ind, f in zip(population, fitnesses):
            if f < best_len:
                best_len = f
                best_tour = ind.copy()

        best_history.append(best_len)

        #  世代ごとの画像を保存 
        order = best_tour + [best_tour[0]]
        xs = cities[order, 0]
        ys = cities[order, 1]

        plt.figure(figsize=(6, 6))
        plt.scatter(cities[:, 0], cities[:, 1])
        for i, (x, y) in enumerate(cities):
            plt.text(x + 20, y + 20, str(i))

        plt.plot(xs, ys, "-o")
        plt.gca().invert_xaxis()  # ジブチ向けの反転
        plt.title(f"Generation {gen}, best={best_len:.2f}")
        plt.grid(True)

        frame_path = f"frames/gen_{gen:04}.png"
        plt.savefig(frame_path)
        plt.close()
        frame_paths.append(frame_path)

        # 次世代生成
        new_pop = [best_tour.copy()]
        while len(new_pop) < pop_size:
            p1 = tournament_select(population, fitnesses)
            p2 = tournament_select(population, fitnesses)

            if random.random() < crossover_rate:
                c1 = crossover_ox(p1, p2)
                c2 = crossover_ox(p2, p1)
            else:
                c1 = p1.copy()
                c2 = p2.copy()

            mutate_swap(c1, mutation_rate)
            mutate_swap(c2, mutation_rate)

            new_pop.append(c1)
            if len(new_pop) < pop_size:
                new_pop.append(c2)

        population = new_pop

    # GIF 生成（frames/*.png → tsp_ga.gif）
    images = [Image.open(f) for f in frame_paths]
    images[0].save(
        "tsp_ga.gif",
        save_all=True,
        append_images=images[1:],
        duration=100,  # ミリ秒
        loop=0
    )
    print("GIF saved as tsp_ga.gif")

    return cities, best_tour, best_len, best_history


# 結果表示
if __name__ == "__main__":
    cities, best_tour, best_len, best_history = ga_tsp(
        num_cities=38,
        pop_size=100,
        generations=500,   # ← GIFなので 500 くらい推奨（大きくしすぎ注意）
        crossover_rate=0.9,
        mutation_rate=0.2,
        seed=0,
    )

    print("Best tour:", best_tour)
    print("Best length:", best_len)


GIF saved as tsp_ga.gif
Best tour: [1, 0, 9, 13, 20, 28, 29, 31, 34, 36, 37, 32, 33, 35, 30, 26, 27, 23, 21, 24, 25, 22, 19, 14, 12, 15, 16, 17, 18, 10, 11, 8, 7, 5, 6, 4, 2, 3]
Best length: 6704.124898413524
